## Criação da pasta para receber arquivos

In [ ]:
import os

nome_da_pasta = 'docs'

if not os.path.exists(nome_da_pasta):
    os.makedirs(nome_da_pasta)
    print(f"Pasta '{nome_da_pasta}' criada com sucesso!")
else:
    print(f"A pasta '{nome_da_pasta}' já existe.")

A pasta 'docs' já existe.


## Instalação e importação de dependências

In [ ]:
!pip install -qqq langchain-text-splitters langchain-huggingface pypdf langchain-chroma langchain-google-genai "transformers<5" torch

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import VectorStoreRetriever
from langchain_core.language_models import BaseChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, Runnable
from pypdf import PdfReader
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from pathlib import Path

import shutil
import torch
import chromadb
import logging
import warnings
import os

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s: %(message)s',
    force=True
)

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.ERROR)

## STEP 1: Criação/carregamento de documento

In [ ]:
def load_document():
    logging.info(f"Loading documents from {SOURCE_DIR}")
    arquivos = sorted(SOURCE_DIR.glob("*.md"))

    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo encontrado em {SOURCE_DIR}")

    documentos = []
    for arquivo in arquivos:
        logging.info(f"  -> {arquivo.name}")
        texto = arquivo.read_text(encoding="utf-8")
        documentos.append(Document(
            page_content=texto,
            metadata={"source": arquivo.name}
        ))

    return documentos

## STEP 2: Criar chunks a partir do documento

In [ ]:
def create_chunks(documents):
    logging.info(f"Creating chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )
    return text_splitter.split_documents(documents)

## STEP 3: Criar embeddings

In [ ]:
def create_embedding_model():
    logging.info(f"Creating embedding model: {EMBEDDINGS_MODEL_NAME}")
    return HuggingFaceEmbeddings(model_name=EMBEDDINGS_MODEL_NAME)

def create_vector_store(chunks, embedding_model):
    logging.info(f"Creating vector store in {VECTOR_STORE_DIR}")
    if Path(VECTOR_STORE_DIR).exists():
        shutil.rmtree(VECTOR_STORE_DIR)

    # Força o Chroma a esquecer o cliente antigo (que apontava pro diretório já deletado)
    chromadb.api.client.SharedSystemClient.clear_system_cache()

    return Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=VECTOR_STORE_DIR,
        collection_name="company_docs"
    )

## STEP 4: Retriever  

In [ ]:
def create_retriever(vector_store):
    return vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": TOP_K}
    )

## STEP 5: Promtp anti-alucinação

In [ ]:
def create_prompt():
    return ChatPromptTemplate.from_template("{context}")

PROMPT_TEMPLATES = {
    "zero_shot": """
    Você é um assistente de suporte técnico da Promob.

    Use APENAS as informações do contexto abaixo para responder. Se a resposta não estiver no contexto, responda exatamente: "Não localizei essa informação no documento."

    Contexto:
    {context}

    Pergunta:
    {input}

    Resposta em português:
    """,

    "few_shot": """
    Você é um assistente de suporte técnico da Promob. Responda com base apenas no contexto fornecido.

    Exemplo:
    Contexto: "Para atualizar o Promob aberto, acesse Ferramentas > Atualizar o Promob, aguarde a busca e clique em Atualizar."
    Pergunta: Como atualizo o Promob com o programa aberto?
    Resposta: Acesse o menu Ferramentas, clique em "Atualizar o Promob" e depois em "Atualizar" quando a verificação terminar.

    Agora responda com o mesmo estilo:
    Contexto: {context}
    Pergunta: {input}
    Resposta:
    """,

    "structured_output": """
    Você é um assistente de suporte técnico da Promob. Responda somente com base no contexto.

    Contexto:
    {context}

    Pergunta:
    {input}

    Responda SOMENTE em formato JSON, sem texto adicional, seguindo este schema:
    {{"resposta": "...", "encontrado_no_documento": true ou false}}
    """
}

def create_prompt():
    logging.info(f"Using prompt strategy: {PROMPT_STRATEGY}")
    return ChatPromptTemplate.from_template(PROMPT_TEMPLATES[PROMPT_STRATEGY])

## STEP 6: Criar LLM

In [ ]:
def create_llm():
    logging.info(f"Loading Hugging Face model: {LLM_MODEL}")

    pipe = pipeline(
        "text2text-generation",
        model=LLM_MODEL,
        device=0 if torch.cuda.is_available() else -1,
        torch_dtype=torch.float16,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        do_sample=True
    )

    return HuggingFacePipeline(pipeline=pipe)

## STEP 7: Criar pipeline RAG

In [ ]:
def create_pipeline(retriever, prompt, llm):
    def format_documents(docs):
        return "\n\n".join([doc.page_content for doc in docs])

    answer_chain = (
        {"context": retriever| format_documents, "input": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return RunnableParallel(answer=answer_chain, sources=retriever)

def create_rag_pipeline():
    documents = load_document()
    chunks = create_chunks(documents)
    model = create_embedding_model()
    vector_store = create_vector_store(chunks, model)
    retriever = create_retriever(vector_store)
    prompt = create_prompt()
    llm = create_llm()
    pipeline = create_pipeline(retriever, prompt, llm)

    return pipeline

## Parâmetros e execução

In [ ]:
# Parâmetros
PROJECT_ROOT = Path("/content/").absolute()
SOURCE_DIR = PROJECT_ROOT / "docs"
VECTOR_STORE_DIR = str(PROJECT_ROOT / "chroma_db")
EMBEDDINGS_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 3
LLM_MODEL = "bigscience/mt0-xl"  # "csebuetnlp/mT5_multilingual_XLSum"
TEMPERATURE = 0.3
MAX_NEW_TOKENS = 256
PROMPT_STRATEGY = "zero_shot"  # "zero_shot" | "few_shot" | "structured_output"

# Perguntas
perguntas = [
    "Como faço para renovar minha assinatura do Promob?", # (Portal - Renovar Assinaturas)
    "Quantos GB de RAM são recomendados para rodar o Promob?", # (Requisitos)
    "É possível atualizar o Promob com o programa fechado?", # (Atualizar)
    "O Promob funciona em máquina virtual?", # (Requisitos, resposta específica dentro de tabela)
    "Qual o preço da licença anual do Promob?" # (Fora do escopo, testa anti-alucinação)
]

def main():
    rag_pipeline = create_rag_pipeline()

    print("\n--- Parâmetros ---")
    print(f"Modelo de embeddings: {EMBEDDINGS_MODEL_NAME}")
    print(f"Tamanho dos chunks: {CHUNK_SIZE}")
    print(f"Overlap: {CHUNK_OVERLAP}")
    print(f"Modelo de LLM: {LLM_MODEL}")
    print(f"Temperatura: {TEMPERATURE}")
    print(f"Máximo de tokens: {MAX_NEW_TOKENS}")
    print(f"Estratégia de prompt: {PROMPT_STRATEGY}")

    print("\n--- Perguntas ---")
    for pergunta in perguntas:
        print(f"\nPergunta: {pergunta}")
        resultado = rag_pipeline.invoke(pergunta)
        print(f"Resposta: {resultado['answer']}")

main()


INFO: Loading documents from /content/docs
INFO:   -> Portal_-_Renovar_Assinaturas_–_Suporte_Promob.md
INFO:   -> Promob_-_Atualizar_o_Promob_–_Suporte_Promob.md
INFO:   -> Promob_-_Requisitos_do_computador_–_Suporte_Promob.md
INFO: Creating chunks (size=500, overlap=50)
INFO: Creating embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
INFO: Creating vector store in /content/chroma_db
INFO: Using prompt strategy: zero_shot
INFO: Loading Hugging Face model: bigscience/mt0-xl


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


--- Parâmetros ---
Modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Tamanho dos chunks: 500
Overlap: 50
Modelo de LLM: bigscience/mt0-xl
Temperatura: 0.3
Máximo de tokens: 256
Estratégia de prompt: zero_shot

--- Perguntas ---

Pergunta: Como faço para renovar minha assinatura do Promob?
Resposta: Acesse o [Portal de Serviços Promob;](https://portal.promob.com) **2.** Acesse sua conta em **_Conecte-se Agora_;** **3.** Clique em _**Assinaturas - Renovar**_ **4\.** Selecione a modalidade a ser renovada (mensal ou anual, dependendo da assinatura);

Pergunta: Quantos GB de RAM são recomendados para rodar o Promob?
Resposta: 4 GB - 8 GB

Pergunta: É possível atualizar o Promob com o programa fechado?
Resposta: Yes

Pergunta: O Promob funciona em máquina virtual?
Resposta: não

Pergunta: Qual o preço da licença anual do Promob?
Resposta: Não localizei essa informação no documento.
